# DanceMatch3 — kostenloser Trainingslauf in Google Colab

Dieses Notebook umgeht GitHub Actions. Es verwendet die bestehende Baseline aus dem Branch `feature/dance-style-training` und läuft im kostenlosen Colab-Runtime, soweit dein Google-Konto dafür Zugriff hat. **Kein GitHub-Actions-Lauf und kein GitHub-Billing nötig.**

Hinweis: Das ist ein erster Forschungs-Baseline-Test, noch kein produktionsreifes Modell für alle neun Tanzstile. Bachata und Discofox brauchen zusätzliche, sauber gelabelte Trainingsdaten.

In [ ]:
# WICHTIG: Auch IPython/Colab-PWD aktualisieren, nicht nur das Python-Arbeitsverzeichnis.%cd /contentimport os, pathlib, shutilos.chdir('/content')os.environ['PWD'] = '/content'REPO_PATH = pathlib.Path('/content/DanceMatch3')if REPO_PATH.exists() and not (REPO_PATH / '.git').exists():    print('Entferne beschädigtes Verzeichnis:', REPO_PATH)    shutil.rmtree(REPO_PATH)print('Arbeitsverzeichnis Python:', os.getcwd())print('PWD für Shell:', os.environ.get('PWD'))# Systembibliotheken und kompatible Python-Abhängigkeiten installieren.# datasets 3.x verwendet fsspec <= 2025.3.0; gcsfs wird passend dazu gepinnt.!apt-get -qq update!apt-get -qq install -y ffmpeg libsndfile1!pip -q install 'datasets>=3.0,<4' 'librosa>=0.10.2,<1' 'numpy>=1.26,<2.3' 'scikit-learn>=1.4,<2' 'joblib>=1.3,<2' 'fsspec==2025.3.0' 'gcsfs==2025.3.0'import datasets, fsspecprint('datasets:', datasets.__version__, '| fsspec:', fsspec.__version__)try:    import gcsfs    print('gcsfs:', gcsfs.__version__)except Exception as e:    print('gcsfs optional nicht verfügbar:', e)# Nicht benötigte Installations-Caches entfernen, damit mehr Colab-Disk frei bleibt.!apt-get clean!rm -rf /var/lib/apt/lists/* /var/cache/apt/archives/*!pip cache purge >/dev/null 2>&1 || truetotal, used, free = shutil.disk_usage('/content')print(f'Colab-Disk nach Setup: {free / (1024**3):.2f} GiB frei von {total / (1024**3):.2f} GiB')print('Setup abgeschlossen')

In [ ]:
# Git-Befehle mit sauberem Python- und Shell-Arbeitsverzeichnis ausführen.
%cd /content
import os, pathlib, shutil, subprocess
BASE = '/content'
REPO = os.path.join(BASE, 'DanceMatch3')
BRANCH = 'feature/dance-style-training'
URL = 'https://github.com/zwirbel/DanceMatch3.git'
os.chdir(BASE)
os.environ['PWD'] = BASE

if os.path.isdir(REPO) and not os.path.isdir(os.path.join(REPO, '.git')):
    print(f'{REPO} existiert, ist aber kein Git-Repository. Entferne es ...')
    shutil.rmtree(REPO)

if os.path.isdir(os.path.join(REPO, '.git')):
    subprocess.run(['git', '-C', REPO, 'fetch', 'origin', BRANCH], cwd=BASE, check=True)
    subprocess.run(['git', '-C', REPO, 'checkout', BRANCH], cwd=BASE, check=True)
    subprocess.run(['git', '-C', REPO, 'pull', '--ff-only', 'origin', BRANCH], cwd=BASE, check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, URL, REPO], cwd=BASE, check=True)

script = pathlib.Path(REPO, 'training', 'train_baseline.py')
if not script.is_file():
    raise FileNotFoundError(f'Trainingsskript nicht gefunden: {script}')
print('Code erfolgreich geladen:', REPO)
print('Branch:', subprocess.run(['git', '-C', REPO, 'branch', '--show-current'], cwd=BASE, check=True, capture_output=True, text=True).stdout.strip())


## Trainingsumfang auswählen

`MAX_ITEMS = 0` verarbeitet alle verfügbaren Beispiele (bis zu 698 Zeilen laut aktuellem Datensatzstand). Eine positive Zahl, zum Beispiel `100`, begrenzt den Lauf auf einen kurzen Test. Der vollständige Lauf kann je nach Colab-Runtime länger dauern.

In [ ]:
MAX_ITEMS = 0  # 0 = vollständiger Lauf; positive Zahl = kurzer FunktionstestCLEAN_DATASET_CACHE_AFTER_DOWNLOAD = True  # spart Disk; Daten werden bei erneutem Training neu geladenDATASET = 'Yi3852/MuCUE'SPLIT = 'ballroom_genres'  # MuCUE hat nur die Dataset-Config 'default'; ballroom_genres ist ein SplitOUT_DIR = f'{REPO}/training/artifacts'os.makedirs(OUT_DIR, exist_ok=True)print('MAX_ITEMS =', MAX_ITEMS, '| DATASET =', DATASET, '| SPLIT =', SPLIT)

In [ ]:
# Baseline trainieren und Kennzahlen berechnen
%cd /content/DanceMatch3/training
!HF_HUB_DISABLE_TELEMETRY=1 python train_baseline.py --dataset "$DATASET" --split "$SPLIT" --max-items "$MAX_ITEMS" --out artifacts


In [ ]:
# Ergebnisdateien anzeigen und als ZIP herunterladenfrom pathlib import Pathimport zipfileartifacts = Path('/content/DanceMatch3/training/artifacts')files = [p for p in artifacts.rglob('*') if p.is_file()]for p in files:    print(p, f'({p.stat().st_size:,} bytes)')if files:    zip_path = '/content/dancematch-training-results.zip'    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:        for p in files:            z.write(p, p.relative_to(artifacts))    from google.colab import files    files.download(zip_path)    # Der ZIP-Export enthält die Ergebnisse; den grossen, erneut ladbaren Dataset-Cache danach freigeben.    if CLEAN_DATASET_CACHE_AFTER_DOWNLOAD:        from datasets import config as datasets_config        cache_path = Path(datasets_config.HF_DATASETS_CACHE)        if cache_path.exists():            cache_bytes = sum(p.stat().st_size for p in cache_path.rglob('*') if p.is_file())            shutil.rmtree(cache_path, ignore_errors=True)            print(f'HF-Dataset-Cache bereinigt: {cache_bytes / (1024**2):.1f} MiB freigegeben.')        else:            print('Kein HF-Dataset-Cache zum Bereinigen gefunden.')        total, used, free = shutil.disk_usage('/content')        print(f'Colab-Disk nach Bereinigung: {free / (1024**3):.2f} GiB frei von {total / (1024**3):.2f} GiB')else:    print('Keine Artefakte gefunden. Bitte Fehlermeldung im Trainingsschritt prüfen.')